In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset
import torch

Load the LLM

In [7]:
# Load model in 4bit (QLoRA)
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-AWQ"   
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # or torch.float16 if needed
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",          
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16, # or float16 if no bf16 support
)

# Prepare model for QLoRA (Necessary)
model = prepare_model_for_kbit_training(model)

AttributeError: 'BitsAndBytesConfig' object has no attribute 'get_loading_attributes'

Prepare the LLM for LORA fine-tuning

In [ ]:
# Set Lora config
lora_config = LoraConfig(
    r=8,                            # Rank, can be tuned
    lora_alpha=16,                  # Alpha, can be tuned
    target_modules=["q_proj", "v_proj"], # Llama-specific (can be changed for others)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# 4. Apply Lora
model = get_peft_model(model, lora_config)

# 5. Load your dataset (example: Alpaca dataset)
dataset = load_dataset("tatsu-lab/alpaca")    # You can replace this with your own dataset
train_data = dataset["train"]

In [ ]:


# 6. Tokenize function
def tokenize_function(example):
    return tokenizer(
        example["instruction"] + " " + example["input"] + " " + example["output"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

tokenized_data = train_data.map(tokenize_function, batched=True, remove_columns=train_data.column_names)

# 7. Set training args
training_args = TrainingArguments(
    output_dir="./qlora-finetuned-llama",
    per_device_train_batch_size=1,     # Batch size 1 to survive 8GB VRAM
    gradient_accumulation_steps=16,    # To simulate bigger batch
    learning_rate=2e-4,
    fp16=True,                         # Use fp16 to save memory
    save_total_limit=2,
    logging_steps=10,
    save_steps=100,
    num_train_epochs=1,                # Start small
    optim="paged_adamw_32bit",          # Memory-efficient optimizer (QLoRA style)
    report_to="none",
)

# 8. Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_data,
    args=training_args,
    tokenizer=tokenizer,
)

# 9. Start Training 🚀
trainer.train()

# 10. Save model
trainer.save_model("./qlora-finetuned-llama")
tokenizer.save_pretrained("./qlora-finetuned-llama")
